# 🌐 Back to Basics: The Standard Deterministic PINN
Before the complexities of Sequence-to-Sequence models and Bayesian uncertainty, Scientific Machine Learning was founded on a beautifully simple idea: **The Standard Physics-Informed Neural Network (PINN)**. 

In this notebook, we return to those roots. We abandon discrete sequences, transformers, and numerical ODE solvers. Instead, we treat space and time as a continuous fabric.

---

## 1. The Mesh-Free Paradigm
Traditional numerical solvers (like Finite Difference or Runge-Kutta) require discretizing time into strict "steps" or a grid. 

A standard PINN is a **mesh-free solver**. It relies on a Multi-Layer Perceptron (MLP) acting as a continuous, universal function approximator:
*   **Inputs:** Continuous coordinates (e.g., time $t$).
*   **Outputs:** The predicted state variables $u(t)$.

Because the neural network is a continuous mathematical function, we can evaluate it at any arbitrary microsecond without needing a grid. To enforce the physics across this continuum, we generate a dense cloud of random **Collocation Points**. The network will be forced to obey the laws of physics at every single one of these random temporal coordinates.

## 2. The Continuous PINN Architecture
We construct a standard Multi-Layer Perceptron (MLP) that maps $t \rightarrow [V, n, m, h]$. 

### The Activation Function Upgrade (SIREN)
Standard PINNs typically use $\tanh$ activations. However, for highly non-linear or oscillatory PDEs like the Hodgkin-Huxley equations, standard activations suffer heavily from **Spectral Bias**—they struggle to learn the high-frequency components of a voltage spike.

To combat this in a standard MLP, we replace $\tanh$ with sine wave activations ($\sin$). This creates a Sinusoidal Representation Network (SIREN), which allows the continuous PINN to capture sharp, high-frequency biological transients much more effectively than standard architectures.

## 3. The Engine: Automatic Differentiation (AutoDiff)
The defining feature of a standard PINN is how it computes physics. We do not use finite differences (like in the PINNsFormer) or an ODE solver (like in PI-NODEs). 

Instead, we use **Automatic Differentiation (AutoDiff)**. The exact same backpropagation engine used to update the network weights is used to calculate the *exact* analytical temporal derivatives of the network's output with respect to its input time $t$. 

### The Deterministic Loss Function
The network is trained to find a single set of optimal weights $\theta$ that minimizes two competing objectives:
$$ \mathcal{L}_{total}(\theta) = \mathcal{L}_{Data}(\theta) + \lambda \mathcal{L}_{PDE}(\theta) $$

**A. The Data Loss ($\mathcal{L}_{Data}$)**
This forces the neural network to anchor itself to reality by matching known sensor data or initial conditions:
$$ \mathcal{L}_{Data} = \frac{1}{N_{data}} \sum_{i=1}^{N_{data}} \| u(t_i) - u_{true}(t_i) \|^2 $$

**B. The AutoDiff Physics Loss ($\mathcal{L}_{PDE}$)**
We pass our random collocation points ($t_c$) through the network, use AutoDiff to find the exact derivative $\frac{du}{dt}$, and plug it into the non-linear physical equation $\mathcal{N}[u]$. If the network obeys physics, this residual will be exactly zero:
$$ \mathcal{L}_{PDE} = \frac{1}{N_{colloc}} \sum_{j=1}^{N_{colloc}} \left\| \frac{du}{dt}(t_j) - \mathcal{N}[u(t_j)] \right\|^2 $$

          ## 4. The Optimization Battle
Standard PINNs are notoriously difficult to optimize because the total loss landscape ($\mathcal{L}_{total}$) is a highly non-convex, rugged mountain range. The optimizer is constantly fighting a tug-of-war between fitting the data points and smoothing out the physics.

### The Local Minima Trap
This exposes the critical weakness of standard deterministic PINNs. When using optimizers like Adam, the network can easily roll into a "valley" (a local minimum) where it thinks it has solved the problem. 

*   **If $\lambda$ is too high:** The network might draw a completely flat line (where $\frac{du}{dt} = 0$, heavily minimizing the physics loss) and completely ignore the biological data.
*   **If $\lambda$ is too low:** It will overfit the data points and act like a standard neural network, ignoring the physical laws in the spaces between the points.

Because the model is deterministic, it will confidently output this wrong answer without providing any metric of uncertainty.

## 5. Continuous Evaluation
Despite the optimization challenges, the ultimate advantage of the standard PINN is seen during inference. 

Because the trained network is just a continuous algebraic function, we do not need to boot up a stiff ODE solver (like Rodas5P or Heun) to evaluate it. We simply generate a highly dense array of continuous time points and pass them through the network in a single forward pass. 

The result is perfectly smooth, mesh-free continuous dynamics evaluated instantly.

In [ ]:
# %% Cell 1: Setup and Continuous Domain Prep
using Lux, SciMLSensitivity, Optimization, OptimizationOptimisers, Statistics, Random, ComponentArrays, Zygote

# Assuming t_train (N,) and z_train (4, N) are loaded.
# Format Data for continuous MLP: Input is 1xN, Output is 4xN
t_data = Float32.(reshape(t_train, 1, :))
z_data = Float32.(z_train)

rng = Random.default_rng()
Random.seed!(rng, 42)

# Generate Collocation Points (e.g., 2000 random points between 0 and 50 ms)
# The network will use AutoDiff at these points to enforce the PDE
N_colloc = 2000
t_min, t_max = minimum(t_train), maximum(t_train)
t_colloc = Float32.(rand(rng, 1, N_colloc) .* (t_max - t_min) .+ t_min)

In [ ]:
# %% Cell 2: Mesh-Free Continuous Architecture

# Build the standard PINN MLP
# Inputs: 1 (Continuous Time, t)
# Outputs: 4 (V, n, m, h)
pinn_model = Lux.Chain(
    Lux.Dense(1 => 32, sin),    # Using 'sin' instead of tanh helps with stiff/spiky data
    Lux.Dense(32 => 64, sin),
    Lux.Dense(64 => 64, sin),
    Lux.Dense(64 => 32, sin),
    Lux.Dense(32 => 4)
)

# Initialize network weights and states
ps, st = Lux.setup(rng, pinn_model)
p_nn = ComponentArray(ps)

In [ ]:
# %% Cell 3: Data and AutoDiff PDE Loss

# (Assumes your standard hh_equations(u) function is loaded)

function physics_loss_continuous(t_c, p)
    # 1. Forward pass on collocation points
    u_pred, _ = pinn_model(t_c, p, st)
    
    # 2. AUTOMATIC DIFFERENTIATION (The core PINN mechanic)
    # We want du/dt for all 4 states. 
    # A standard Zygote trick for batch 1D gradients: taking the gradient of the 
    # sum of outputs w.r.t inputs gives the exact element-wise derivatives.
    du_dt = vcat([
        Zygote.gradient(t -> sum(pinn_model(t, p, st)[1][i, :]), t_c)[1] 
        for i in 1:4
    ]...)
    
    # 3. Calculate true HH dynamics at these predicted states
    true_dynamics = hh_equations(u_pred)
    
    # 4. PDE Residual: The difference between AutoDiff derivative and Physical derivative
    # (Assuming scale_factors is available from your previous scaling code)
    residuals = (du_dt .- true_dynamics) ./ scale_factors
    
    return mean(abs2, residuals)
end

function total_loss(p, _)
    # --- A. Data Loss ---
    # Enforce that the network matches our known sensor readings
    u_data_pred, _ = pinn_model(t_data, p, st)
    loss_data = mean(abs2, u_data_pred .- z_data)
    
    # --- B. Physics Loss ---
    # Enforce that the network obeys the PDE at random collocation points
    loss_phys = physics_loss_continuous(t_colloc, p)
    
    # Balance the two losses (often requires heavy tuning in standard PINNs)
    λ = 1.0f0 
    
    return loss_data + λ * loss_phys
end

In [ ]:
# %% Cell 4: Training

loss_history = Float32[]

callback_fn = function (p, l)
    push!(loss_history, l)
    if length(loss_history) % 100 == 0
        println("Epoch $(length(loss_history)) | Total Loss: $(round(l, digits=5))")
    end
    return false 
end

opt_func = OptimizationFunction(total_loss, Optimization.AutoZygote())
opt_prob = OptimizationProblem(opt_func, p_nn)

println("Starting Standard PINN Training...")
# PINNs usually require many epochs due to the complex loss landscape
result = solve(opt_prob, Adam(0.001), maxiters = 5000, callback = callback_fn)
println("Training Complete!")

p_opt = result.u

In [ ]:
# %% Cell 5: Evaluation and Plotting
using Plots

# Create a highly dense, continuous time grid for smooth plotting
t_high_res = Float32.(reshape(range(minimum(t_train), maximum(t_train), length=1000), 1, :))

# Get the network's continuous predictions
u_pred_high_res, _ = pinn_model(t_high_res, p_opt, st)

# Plot Voltage Comparison
plot(t_high_res[1,:], u_pred_high_res[1,:], 
    label="PINN Prediction", 
    linewidth=2, 
    color=:cyan,
    title="Standard Continuous PINN Prediction",
    xlabel="Time",
    ylabel="Voltage"
)

# Overlay the true data points
scatter!(t_data[1,:], z_data[1,:], 
    label="True Data", 
    markersize=2, 
    color=:orange,
    alpha=0.5
)